# 6.3 怎麼訓練實用的 tokenizer？


如果只把訓練見過的中文放進字表，新人名或emoji可能沒法編碼。[6.1](https://birdhackor.github.io/tiny-perceptron-vlm/6.1.html) 已告訴我們任何UTF-8文字都能先化成0到255的byte；[6.2](https://birdhackor.github.io/tiny-perceptron-vlm/6.2.html) 則讓常見相鄰項合併。把兩者接起來，先保證所有byte可用，再讓頻繁片段佔一個token，就得到byte-level BPE。

本節使用已經隨專案安裝的tokenizers套件，實現統計與保存細節；訓練材料仍由我們直接寫兩句，不下載網絡資料。它的「訓練」只計出token拆分表，不會產生會回答問題的模型。後面使用這張表訓練語言模型時，輸入輸出詞表大小與ID含義必須匹配。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers, decoders

tok = Tokenizer(models.BPE())
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(
    vocab_size=280,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    show_progress=False,
)
tok.train_from_iterator(["貓看狗。", "a cat sees a dog."], trainer)
text = "小鳥🙂new"
encoded = tok.encode(text)
print("片段", encoded.tokens)
print("ID", encoded.ids)
print("還原", tok.decode(encoded.ids))
assert tok.decode(encoded.ids) == text

`models.BPE` 選合併規則類型；`ByteLevel` 前處理把byte映射為工具內部可見符號，再給BPE統計。`initial_alphabet` 納入完整256種起始byte符號，而非只加入兩句出現的字符。目標詞表280留下少量合併空間；數據很少時未必能恰好長到目標上限。`add_prefix_space=False` 避免主動在開頭加空格，也沒有額外normalizer改原文。

decoder是反方向的解碼工具，按ByteLevel約定拼回bytes再轉UTF-8。輸入新句含訓練未見的小、鳥與🙂，它們仍能以細片段表示。印出的tokens可能像奇怪拉丁符號，因為展示的是byte的可見代號，不是損壞的原句；應核對最後「還原」行完整為 `小鳥🙂new` 與assert通過，而不要求每token肉眼可讀成完整字。

訓練字表至少要容納256種byte，再加需要的特殊項與合併項。把目標設得很小，並不會憑空獲得完整覆蓋又大量壓縮；工具的實際字表大小也要檢查。這裡兩句資料與280只是短演示，不是自然中文語料的推薦配置。

最後要把tokenizer保存、隨模型一起記錄版本，例如 `tok.save("tokenizer.json")` 可以存規則，但本程式不寫出永久檔。重新訓練同樣大小的詞表，ID也可能對到不同片段，不能直接裝到舊模型上當同一語言接口。

練習只把text改成 `"未見字🦊"`，先預測可能切得較細，但decode仍等於原文，再執行核對。若你同時改了前處理規則，應重新驗證空格、換行與特殊字符；還原契約來自完整規則組合，不只BPE這個名字。
